# LightGBM - ใช้เทียบกับ top-down

In [ ]:
# %% 1) Config + load
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

DATA_PATH = "train.csv"   # วาง train.csv ไว้โฟลเดอร์เดียวกับ notebook
ANCHORS = pd.date_range("2015-01-01", "2017-10-01", freq="QS")
PARAMS = {"num_leaves": 31, "learning_rate": 0.03, "min_child_samples": 50, "n_estimators": 1000}  # ชุดเดิม ไม่ได้ tune ใหม่
FEATURES = ["store", "item", "lag_1", "lag_7", "lag_14", "lag_28", "lag_365",
            "rmean_7", "rmean_14", "rmean_28", "rmean_90", "rstd_7", "rstd_28", "rstd_90",
            "cv_7", "cv_28", "dow", "month", "month_sin", "month_cos", "item_avg", "store_avg"]
MIN_HIST = 365  # เท่ากับ dropna เดิม (lag_365)

raw = pd.read_csv(DATA_PATH, parse_dates=["date"])
Y_df = raw.pivot_table(index=["store", "item"], columns="date", values="sales")
DATES = Y_df.columns
DATES_EXT = DATES.append(pd.date_range("2018-01-01", "2018-03-31"))   # วันใน horizon จริง
SERIES = Y_df.index.to_frame(index=False)
STORE, ITEM = SERIES["store"].to_numpy(), SERIES["item"].to_numpy()
Y_all = Y_df.to_numpy(float)
S = len(SERIES)
print(Y_all.shape)

In [2]:
# %% 2) Features: feature ของวัน index ts ทุก series ใช้เฉพาะ Y[:, :t]
def features(Y, ts):
    ts = np.asarray(ts)
    C = np.concatenate([np.zeros((S, 1)), np.cumsum(Y, axis=1)], axis=1)        # C[:, t] = sum Y[:, :t]
    C2 = np.concatenate([np.zeros((S, 1)), np.cumsum(Y ** 2, axis=1)], axis=1)
    out = {}
    for k in (1, 7, 14, 28, 365):
        out[f"lag_{k}"] = Y[:, ts - k]
    for w in (7, 14, 28, 90):
        s1 = C[:, ts] - C[:, ts - w]
        out[f"rmean_{w}"] = s1 / w
        if w != 14:
            s2 = C2[:, ts] - C2[:, ts - w]
            out[f"rstd_{w}"] = np.sqrt(np.maximum(s2 - s1 ** 2 / w, 0) / (w - 1))
    out["cv_7"] = out["rstd_7"] / (out["rmean_7"] + 1e-5)
    out["cv_28"] = out["rstd_28"] / (out["rmean_28"] + 1e-5)
    # ค่าเฉลี่ยยอดขายต่อวันของ item (ทุก store) / store (ทุก item) จากทุกวันก่อนวัน t
    item_sum = pd.DataFrame(C[:, ts]).groupby(ITEM).transform("sum").to_numpy()
    store_sum = pd.DataFrame(C[:, ts]).groupby(STORE).transform("sum").to_numpy()
    out["item_avg"] = item_sum / (ts * STORE.max())
    out["store_avg"] = store_sum / (ts * ITEM.max())
    d = DATES_EXT[ts]
    n = len(ts)
    df = pd.DataFrame({k: v.ravel() for k, v in out.items()})   # เรียง series ก่อน แล้ววัน
    df["store"] = pd.Categorical(np.repeat(STORE, n), categories=sorted(set(STORE)))
    df["item"] = pd.Categorical(np.repeat(ITEM, n), categories=sorted(set(ITEM)))
    df["dow"] = np.tile(d.dayofweek, S)
    df["month"] = np.tile(d.month, S)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df[FEATURES]


def smape(a, f):
    denom = np.abs(a) + np.abs(f)
    return 100 * np.mean(np.where(denom == 0, 0, 2 * np.abs(f - a) / denom))


def fit(Y):
    tr_ts = np.arange(MIN_HIST, Y.shape[1])
    model = lgb.LGBMRegressor(**PARAMS, force_row_wise=True, n_jobs=-1, random_state=42,
                              deterministic=True, verbose=-1)
    model.fit(features(Y, tr_ts), Y[:, tr_ts].ravel(), categorical_feature=["store", "item"])
    return model


def recursive(model, Y, h, buggy=False):
    """ทายทีละวัน แล้วต่อท้าย Y; buggy=True จำลอง bug เดิม (ใช้ feature ของวัน t-1 ทายวัน t)"""
    for _ in range(h):
        t = Y.shape[1]
        if buggy:
            X = features(Y, [t - 1])
        else:
            X = features(np.concatenate([Y, np.zeros((S, 1))], axis=1), [t])  # ช่องของวัน t ไม่ถูกใช้
        Y = np.concatenate([Y, model.predict(X)[:, None]], axis=1)
    return Y[:, -h:]

In [ ]:
# %% 3) Backtest 12 anchors
rows, saved = [], []
for anchor in ANCHORS:
    t0 = time.time()
    a = DATES.get_loc(anchor)
    end = DATES.get_loc(anchor + pd.DateOffset(months=3) - pd.Timedelta(days=1)) + 1
    model = fit(Y_all[:, :a])
    pred = recursive(model, Y_all[:, :a].copy(), end - a)
    act, p = Y_all[:, a:end], np.clip(pred, 0, None)
    rows.append({"anchor": anchor, "model": "LightGBM_fixed", "SMAPE": smape(act, p),
                 "RMSE": float(np.sqrt(np.mean((p - act) ** 2))), "MAE": float(np.mean(np.abs(p - act)))})
    saved.append(pd.DataFrame({
        "unique_id": np.repeat(SERIES["store"].astype(str) + "_" + SERIES["item"].astype(str), end - a),
        "ds": np.tile(DATES[a:end], S), "y": act.ravel(), "LightGBM": pred.ravel(), "anchor": anchor}))
    print(f"{anchor.date()} SMAPE={rows[-1]['SMAPE']:.2f} ({time.time() - t0:.0f}s)")

lgbm_results = pd.DataFrame(rows)
lgbm_results.to_csv("lgbm_fixed_backtest_results.csv", index=False)
pd.concat(saved, ignore_index=True).to_csv("lgbm_fixed_backtest_preds.csv", index=False)
print(lgbm_results[["SMAPE", "RMSE", "MAE"]].agg(["mean", "std"]).round(3))

2015-01-01 SMAPE=15.71 (5s)


2015-04-01 SMAPE=11.99 (7s)


2015-07-01 SMAPE=11.73 (7s)


2015-10-01 SMAPE=13.38 (8s)


2016-01-01 SMAPE=13.92 (9s)


2016-04-01 SMAPE=11.62 (10s)


... (3 outputs left out)

2017-04-01 SMAPE=11.19 (15s)


2017-07-01 SMAPE=11.03 (16s)


2017-10-01 SMAPE=12.70 (19s)


       SMAPE   RMSE    MAE
mean  12.628  7.850  6.057
std    1.401  0.575  0.443


In [ ]:
# %% 4) เทียบกับ model ใน demand_forecast_topdown.ipynb (backtest เดียวกัน)
td = pd.read_csv("backtest_results.csv", parse_dates=["anchor"])
allres = pd.concat([td, lgbm_results], ignore_index=True)
summary = allres.groupby("model")[["SMAPE", "RMSE", "MAE"]].mean().sort_values("SMAPE").round(3)
summary["SMAPE_jan"] = allres[allres["anchor"].dt.month == 1].groupby("model")["SMAPE"].mean().round(2)
summary

In [ ]:
# %% 5) Forecast 2018-Q1 ด้วย LightGBM 
model = fit(Y_all)
pred = np.clip(recursive(model, Y_all.copy(), 90), 0, None)
fc_lgbm = pd.DataFrame({"store": np.repeat(STORE, 90), "item": np.repeat(ITEM, 90),
                        "date": np.tile(pd.date_range("2018-01-01", "2018-03-31"), S), "sales": pred.ravel()})
assert fc_lgbm.shape == (45000, 4) and fc_lgbm["sales"].notna().all()
fc_lgbm.to_csv("forecast_2018_LightGBM_fixed.csv", index=False)

ens = pd.read_csv("forecast_2018_TopDown_Ens.csv", parse_dates=["date"])
cmp = (ens.groupby(ens["date"].dt.month)["sales"].sum().rename("TopDown_Ens").to_frame()
       .join(fc_lgbm.groupby(fc_lgbm["date"].dt.month)["sales"].sum().rename("LightGBM_fixed")))
cmp["LightGBM / Ens"] = cmp["LightGBM_fixed"] / cmp["TopDown_Ens"]
cmp.round(3)

,TopDown_Ens,LightGBM_fixed,LightGBM / Ens
date,,,
1,656148.565,653961.836,0.997
2,667266.934,655368.031,0.982
3,889755.702,875069.725,0.983
